In [1]:
import torch
import sys
import time

sys.path.append("/home/atuin/v120bb/v120bb18/UnReflectAnything")
from utilities.visualization import rgb, panelize
from polar_highlighter import PolarHighlighter

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    curr_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(curr_device)
    print(f"CUDA is available: {num_devices} device(s) detected.")
    print(f"Current device id: {curr_device} - {device_name}")
else:
    print("CUDA is not available")
%load_ext autoreload
%autoreload 2


In [2]:
MODELS = ["icy-gorge-754", "easy-surf-759"]

In [3]:
from main import load_and_process_config
from models_utils import load_best_model_by_run
from dataset import from_config
from utilities import tensor_dict_summarize


config = load_and_process_config("config_train.yaml")
config.RUN = "easy-surf-759"
# config.DATASETS = {"PSD": config.DATASETS.PSD}

dataset = from_config(config)["validation"]
idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False))
model = load_best_model_by_run(config.RUN).eval()

In [4]:
METHODS = [
    "DHAN-SHR_output",
    "PolarFree_output",
    "SpecularityNet_output",
    "TSHRNet_output",
    "TSMSHD_output",
    "EndoSTTN_output",
    "StableDelight_output",
]

In [5]:
from torch.utils.data.dataloader import default_collate
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
import importlib
import models
import random

importlib.reload(models)
from inference import list_image_paths
from PIL import Image
from torchvision.transforms import functional as TF

config = load_and_process_config("config_train.yaml")
config.RUN = "easy-surf-759"
model = load_best_model_by_run(config.RUN).eval()

# Get target size from model config
target_side = model.dinov3.config["image_size"]
target_size = (target_side, target_side)



In [6]:

class ImageDatasetCropResize(Dataset):
    """Dataset for loading images with center crop and resize (maintaining aspect ratio)."""

    def __init__(self, paths: list, target_size: tuple):
        """
        Args:
            paths: List of image file paths
            target_size: Target size (H, W) for output images
        """
        self.paths = paths
        self.target_size = target_size  # (H, W)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        """Load and preprocess a single image with center crop and resize.

        Returns:
            Tuple of (image_tensor [3, H, W], original_size [2], path_string)
        """
        path = self.paths[idx]
        with Image.open(path) as img:
            rgb_img = img.convert("RGB")
            original_size = rgb_img.size[::-1]  # PIL size is (W, H), we need (H, W)

            # Convert to tensor: [3, H, W]
            tensor = TF.to_tensor(rgb_img)

            # Center crop to square (maintain aspect ratio)
            _, h, w = tensor.shape
            min_dim = min(h, w)
            # Calculate crop coordinates (center crop)
            top = (h - min_dim) // 2
            left = (w - min_dim) // 2
            tensor = TF.crop(tensor, top, left, min_dim, min_dim)

            # Resize to target size
            tensor = TF.resize(tensor, self.target_size, antialias=True)

        # Convert size to tensor for proper batching: [H, W]
        size_tensor = torch.tensor(original_size, dtype=torch.int32)
        return tensor, size_tensor, str(path)


# Handle image directory: can be direct path or parent directory with subdirectories
image_dir = Path("/anvme/workspace/v120bb18-unreflectanything/benchmark/data/input")
# Alternative: parent directory with subdirectories (randomly selects one)
image_dir = Path("/anvme/workspace/v120bb18-unreflectanything/benchmark/data/input")

image_extensions = [".png", ".jpg", ".jpeg"]

# Check if image_dir contains image files directly (non-recursive check)
direct_image_files = [
    p
    for p in image_dir.iterdir()
    if p.is_file() and p.suffix.lower() in [ext.lower() for ext in image_extensions]
]

if direct_image_files:
    # Direct path to images - use list_image_paths which does recursive search
    image_paths = list_image_paths(image_dir, image_extensions)
    print(f"Loading {len(image_paths)} images directly from: {image_dir}")
else:
    # Parent directory with subdirectories - randomly select from subdirectories
    subdirs = [d for d in image_dir.iterdir() if d.is_dir()]
    if not subdirs:
        raise ValueError(
            f"No image files or subdirectories found in {image_dir}. "
            f"Please check the path or ensure it contains images or dataset subdirectories."
        )

    # Randomly select a subdirectory
    selected_subdir = random.choice(subdirs)
    image_paths = list_image_paths(selected_subdir, image_extensions)
    print(f"Randomly selected subdirectory: {selected_subdir}")
    print(f"Loading {len(image_paths)} images from: {selected_subdir}")

# Create ImageDataset with crop+resize
image_dataset = ImageDatasetCropResize(image_paths, target_size)


# Custom collate function that handles ImageDataset output format
# ImageDataset returns: (image_tensor [3, H, W], size_tensor [2], path_string)
def collate_with_paths(batch):
    # batch is a list of tuples: [(image, size, path), ...]
    images = [item[0] for item in batch]  # List of [3, H, W] tensors
    sizes = [item[1] for item in batch]  # List of [2] tensors
    paths = [item[2] for item in batch]  # List of strings

    # Stack images: [B, 3, H, W]
    images_batch = torch.stack(images, dim=0)
    # Stack sizes: [B, 2]
    sizes_batch = torch.stack(sizes, dim=0)

    # Print path (batch size is always 1)
    image_path = paths[0]
    print(f"Loaded image path: {image_path}")

    return {
        "raw": images_batch,  # [B, 3, H, W] - using "raw" to match cell 5 expectations
        "rgb": images_batch,  # [B, 3, H, W] - also provide "rgb" for compatibility
        "original_size": sizes_batch,  # [B, 2]
        "path": image_path,  # Single path string (batch size 1)
        "filepaths": {  # Match expected format from original dataset
            "raw_path": [image_path]  # List with single path for compatibility
        },
    }


# Create DataLoader with batch_size=1
dataloader = DataLoader(
    image_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_with_paths,
    num_workers=0,  # Set to 0 for notebook compatibility
    pin_memory=torch.cuda.is_available(),
)


In [10]:
ls /anvme/workspace/v120bb18-unreflectanything/benchmark/data/input

In [ ]:
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True,    collate_fn=collate_with_paths,)

for d in range(13):
    image_dir = Path("/anvme/workspace/v120bb18-unreflectanything/benchmark/data/input")

    image_extensions = [".png", ".jpg", ".jpeg"]

    # Check if image_dir contains image files directly (non-recursive check)
    direct_image_files = [
        p
        for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in [ext.lower() for ext in image_extensions]
    ]

    if direct_image_files:
        # Direct path to images - use list_image_paths which does recursive search
        image_paths = list_image_paths(image_dir, image_extensions)
        print(f"Loading {len(image_paths)} images directly from: {image_dir}")
    else:
        # Parent directory with subdirectories - randomly select from subdirectories
        subdirs = [d for d in image_dir.iterdir() if d.is_dir()]
        if not subdirs:
            raise ValueError(
                f"No image files or subdirectories found in {image_dir}. "
                f"Please check the path or ensure it contains images or dataset subdirectories."
            )

        # Randomly select a subdirectory
        selected_subdir = random.choice(subdirs)
        image_paths = list_image_paths(selected_subdir, image_extensions)
        print(f"Randomly selected subdirectory: {selected_subdir}")
        print(f"Loading {len(image_paths)} images from: {selected_subdir}")

    # Create ImageDataset with crop+resize
    image_dataset = ImageDatasetCropResize(image_paths, target_size)
    # Create DataLoader with batch_size=1
    dataloader = DataLoader(
        image_dataset,
        batch_size=1,
        shuffle=True,
        collate_fn=collate_with_paths,
        num_workers=0,  # Set to 0 for notebook compatibility
        pin_memory=torch.cuda.is_available(),
    )

    for b, batch in enumerate(dataloader):
        batch = {
            k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
            for k, v in batch.items()
        }
        patch_mask = (
            batch["raw"].mean(dim=1, keepdim=True)
            > 0.9 * batch["raw"].mean(dim=1).max().item()
        )
        model_input = {"rgb": batch["raw"], "inpaint_mask_override": patch_mask}
        # model_input = {"rgb": batch["raw"]}

        with torch.no_grad():
            modelout = model(model_input)
        # rgb(batch["raw"][0], as_tensor=False, resize=(448, 448)),
        # rgb(modelout["diffuse"][0], as_tensor=False, resize=(448, 448)),
        filepath = batch["filepaths"]["raw_path"][0]
        
        # Load method outputs from corresponding directories
        # Path structure: /path/to/benchmark/data/input/DATASET_NAME/filename.png
        # Method paths: /path/to/benchmark/data/METHOD_NAME/DATASET_NAME/filename.png
        # Note: Filenames may differ, so we find files by position in natural ordering
        method_images = []
        input_path = Path(filepath)
        
        path_str = str(input_path)
        if "/data/input/" in path_str:
            # Get the input directory and find the index of current file in natural order
            input_dir = input_path.parent
            # Get all image files in natural sorted order
            image_extensions = [".png", ".jpg", ".jpeg"]
            input_files = sorted([
                f for f in input_dir.iterdir() 
                if f.is_file() and f.suffix.lower() in [ext.lower() for ext in image_extensions]
            ])
            
            # Find the index of the current file
            try:
                file_index = input_files.index(input_path)
            except ValueError:
                file_index = None
                print(f"Warning: Could not find current file in directory listing: {filepath}")
            
            if file_index is not None:
                # Load images from each method directory using the same index
                for method_name in METHODS:
                    method_path_str = path_str.replace("/data/input/", f"/data/{method_name}/")
                    method_dir = Path(method_path_str).parent
                    
                    if method_dir.exists():
                        # Get all image files in method directory in natural sorted order
                        method_files = sorted([
                            f for f in method_dir.iterdir() 
                            if f.is_file() and f.suffix.lower() in [ext.lower() for ext in image_extensions]
                        ])
                        
                        if file_index < len(method_files):
                            method_path = method_files[file_index]
                            # Load image and convert to tensor
                            with Image.open(method_path) as img:
                                method_img = img.convert("RGB")
                                method_tensor = TF.to_tensor(method_img)
                                # Center crop and resize to match input processing
                                _, h, w = method_tensor.shape
                                min_dim = min(h, w)
                                top = (h - min_dim) // 2
                                left = (w - min_dim) // 2
                                method_tensor = TF.crop(method_tensor, top, left, min_dim, min_dim)
                                method_tensor = TF.resize(method_tensor, (448, 448), antialias=True)
                            method_images.append(method_tensor)
                        else:
                            # If index is out of range, create a black image placeholder
                            method_tensor = torch.zeros(3, 448, 448)
                            method_images.append(method_tensor)
                            print(f"Warning: Method directory has fewer files ({len(method_files)}) than input directory. Index {file_index} out of range for: {method_dir}")
                    else:
                        # If directory doesn't exist, create a black image placeholder
                        method_tensor = torch.zeros(3, 448, 448)
                        method_images.append(method_tensor)
                        print(f"Warning: Method directory not found: {method_dir}")
            else:
                # If we couldn't find the file index, create black placeholders for all methods
                for method_name in METHODS:
                    method_tensor = torch.zeros(3, 448, 448)
                    method_images.append(method_tensor)
        else:
            print(f"Warning: Could not parse path structure. Expected '/data/input/' in path: {filepath}")
            # Create black placeholders for all methods
            for method_name in METHODS:
                method_tensor = torch.zeros(3, 448, 448)
                method_images.append(method_tensor)
        
        # Build panel with input, our output, and all method outputs
        panel_images = [
            rgb(batch["raw"][0], as_tensor=True, resize=(224,224),border={"color":"#ffffff","thickness":2}),
        ]
        # Add method outputs
        for method_tensor in method_images:
            panel_images.append(rgb(method_tensor, as_tensor=True, resize=(224,224),border={"color":"#ffffff","thickness":2}))
        
        panel_images.append(rgb(modelout["diffuse"][0], as_tensor=True, resize=(224,224),border={"color":"#ffffff","thickness":2}))
        rgb(
            panelize(*panel_images)
        )
        if b > 1:
            break

In [ ]:
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(448, 448),
        blackout=True,
        return_pca=True,
    )
    
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(448, 448),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(448, 448),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
            rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448)),
        )
    )
    if b > 3:
        break


In [5]:
config = load_and_process_config("config_train.yaml")
config.RUN = "curious-moon-751"
config.DATASETS = {"PSD": config.DATASETS.PSD}
dataset = from_config(config)["validation"]
idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False))
model = load_best_model_by_run(config.RUN).eval()
model.token_inpaint._prior_kernel = 5


In [6]:
print(model.token_inpaint._prior_kernel)

In [ ]:
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for b, batch in enumerate(dataloader):
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
        for k, v in batch.items()
    }

    patch_mask = batch["raw"].mean(dim=1, keepdim=True) > 0.8
    model_input = {"rgb": batch["raw"]}  # , "inpaint_mask_override": patch_mask}

    with torch.no_grad():
        modelout = model(model_input)
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(448, 448),
        blackout=True,
        return_pca=True,
    )
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(448, 448),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(448, 448),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
            rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448)),
        )
    )
    if b > 3:
        break


In [ ]:
config = load_and_process_config("config_train.yaml")
config.RUN = "curious-moon-751"

# config.DATASETS = {"PSD": config.DATASETS.PSD}
# dataset = from_config(config)["validation"]
# idataloadr = iter(torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False))
model = load_best_model_by_run(config.RUN).eval()
# model.token_inpaint._prior_kernel = 11

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for b, batch in enumerate(dataloader):
    batch = {
        k: v.cuda() if isinstance(v, torch.Tensor) and torch.cuda.is_available() else v
        for k, v in batch.items()
    }

    patch_mask = batch["raw"].mean(dim=1, keepdim=True) > 0.8
    model_input = {"rgb": batch["raw"]}  # , "inpaint_mask_override": patch_mask}

    with torch.no_grad():
        modelout = model(model_input)
    _, pca = rgb(
        modelout["tokens_completed"][-1].reshape(1, 28, 28, 1024).permute(0, 3, 1, 2),
        as_tensor=True,
        resize=(448, 448),
        blackout=True,
        return_pca=True,
    )
    rgb(
        panelize(
            rgb(batch["raw"][0], as_tensor=True, resize=(448, 448)),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2),
            #     as_tensor=True,
            #     pca=pca,
            #     resize=(448, 448),
            # ),
            # rgb(
            #     modelout["tokens_completed"][-1]
            #     .reshape(1, 28, 28, 1024)
            #     .permute(0, 3, 1, 2)
            #     * torch.logical_not(modelout["patch_mask"].reshape(1, 1, 28, 28)),
            #     as_tensor=True,
            #     resize=(448, 448),
            #     pca=pca,
            #     blackout=True,
            # ),
            rgb(modelout["diffuse"][0], as_tensor=True, resize=(448, 448)),
            rgb(modelout["highlight"][0], as_tensor=True, resize=(448, 448)),
        )
    )
    if b > 3:
        break


In [ ]:
# torch.save(model.token_inpaint.state_dict(), "/home/atuin/v120bb/v120bb18/UnReflectAnything/weights/token_inpainter.pth")